# 08: Qwen3-8B 비추론 비증강 QLoRA 평가, Colab

07의 대응 노트북과 같은 모델, 프롬프트 및 파라미터를 사용하며 학습 증강 확률은 0.0입니다. `prefix_path`는 08의 데이터 디렉터리를 가리킵니다. 상단 딕셔너리를 확인한 뒤 순서대로 실행합니다. 다른 실험의 데이터와 어댑터를 읽지 않습니다.

07의 실제 실행과 비교할 때는 `train_desktop_unsloth.ipynb`로 학습하고 `evaluate_desktop.ipynb`로 평가합니다. 일반 학습 및 Colab 경로는 별도 실행 구성으로 보존하며 백엔드와 환경 차이를 증강 효과에 섞어 해석하지 않습니다. 자세한 실행 및 비교 조건은 [실험 안내](README.md)를 따릅니다.


## 1. 파라미터와 환경


In [ ]:
RUNTIME_PARAMS = {'prefix_path': '/drive/MyDrive/SiteBoard/08/',
 'model_name': 'Qwen/Qwen3-8B',
 'model_revision': 'b968826d9c46dd6066d109eabc6255188de91218',
 'adapter_path': 'models',
 'test_file': 'test.jsonl',
 'split_meta_file': 'split_meta.json',
 'run_config_file': 'result/run_config.json',
 'report_file': 'result/evaluation.json',
 'device': 'cuda'}

HYPER_PARAMS = {'chat_template': {'enable_thinking': False},
 'quantization': {'load_in_4bit': True,
                  'bnb_4bit_quant_type': 'nf4',
                  'bnb_4bit_use_double_quant': True},
 'precision': {'compute_dtype': 'auto', 'frozen_embedding_dtype': 'auto'},
 'model_loading': {'device_map': {'': 0}, 'attn_implementation': 'sdpa', 'low_cpu_mem_usage': True},
 'tokenizer': {'padding_side': 'left'},
 'evaluation': {'batch_size': 1,
                'max_new_tokens': 512,
                'do_sample': False}}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 06과 같이 Colab 설치 셀에서 패키지를 설치합니다. 버전 범위는 루트 requirements.txt와 같습니다.
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", *['torch>=2.4.0', 'transformers>=4.57.6,<5', 'hf_xet>=1.1.3,<2', 'peft>=0.17.1,<0.18', 'accelerate>=0.29.0,<2', 'bitsandbytes>=0.50.2,<1', 'tqdm>=4.66.0', 'ipykernel>=6.29.0,<7', 'ipywidgets>=8,<9']])


## 2. 라이브러리


In [ ]:
import hashlib
import json
import math
import platform
import re
import sys
import time
from collections import Counter
from copy import deepcopy
from pathlib import Path

import torch
from peft import PeftModel
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


## 3. 경로와 파일 해시


In [ ]:
def runtime_path(field):
    return str(Path(RUNTIME_PARAMS["prefix_path"]) / RUNTIME_PARAMS[field])


def write_json(path, value, mode="x"):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open(mode, encoding="utf-8") as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2)
        stream.write("\n")


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def resolve_dtype(name):
    if name == "auto":
        return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return getattr(torch, name)


## 4. 평가 데이터 읽기


In [ ]:
def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]



## 5. 엄격한 JSON 채점과 값 토큰 신뢰도


In [ ]:
def unique_object(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"중복 JSON 키: {key}")
        result[key] = value
    return result


def parse_prediction(text):
    """필드와 값 타입을 검사하고, 유효하지 않은 출력은 None으로 반환합니다."""
    try:
        data = json.loads(text, object_pairs_hook=unique_object)
        if not isinstance(data, list):
            return None
        for item in data:
            if not isinstance(item, dict) or set(item) != {"workType", "spec", "quantity"}:
                return None
            if not isinstance(item["workType"], str) or not item["workType"].strip():
                return None
            if not isinstance(item["spec"], str):
                return None
            item["workType"].encode("utf-8")
            item["spec"].encode("utf-8")
            if type(item["quantity"]) not in (int, float) or not math.isfinite(item["quantity"]):
                return None
        return data
    except (ValueError, TypeError, OverflowError, RecursionError):
        return None


def parse_and_normalize(text):
    data = parse_prediction(text)
    if data is None:
        return None
    return [(item["workType"].strip(), item["spec"].strip(), round(item["quantity"], 4)) for item in data]


def build_token_spans(tokenizer, token_ids, probabilities, pred_text):
    """Qwen 바이트 토큰을 복원해 한글 분할 토큰에도 같은 값 구간을 연결합니다."""
    visible = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
    missing = [value for value in range(256) if value not in visible]
    byte_decoder = dict(zip(map(chr, visible + list(range(256, 256 + len(missing)))), visible + missing))
    spans = []
    restored = bytearray()
    try:
        for token, probability in zip(tokenizer.convert_ids_to_tokens(token_ids), probabilities):
            start = len(restored)
            restored.extend(byte_decoder[char] for char in token)
            spans.append((start, len(restored), probability))
    except KeyError:
        return None
    if len(spans) != len(token_ids) or bytes(restored) != pred_text.encode("utf-8"):
        return None
    return spans


def extract_values_with_confidence(pred_text, token_spans):
    """값과 겹치는 생성 토큰을 집계하며 대응이 미확인인 값은 None으로 둡니다."""
    data = parse_prediction(pred_text)
    if data is None:
        return [], None, None
    predictions = [dict(item) for item in data]
    for item in predictions:
        for field in ("workType", "spec", "quantity"):
            item[f"{field}_confidence"] = None
    if token_spans is None:
        return predictions, None, None

    # JSON 문자열 전체를 먼저 매칭해 값 내부의 키처럼 보이는 문자열을 제외합니다.
    string = r'"(?:[^"\\\x00-\x1f]|\\(?:["\\/bfnrt]|u[0-9a-fA-F]{4}))*"'
    number = r'-?(?:0|[1-9][0-9]*)(?:\.[0-9]+)?(?:[eE][+-]?[0-9]+)?'
    fields = list(re.finditer(rf'({string})\s*:\s*({string}|{number})|{string}', pred_text))
    fields = [match for match in fields if match.group(1) is not None]
    if len(fields) != 3 * len(data):
        return predictions, None, None
    byte_offsets = [0]
    for char in pred_text:
        byte_offsets.append(byte_offsets[-1] + len(char.encode("utf-8")))
    value_indices = set()
    for index, match in enumerate(fields):
        field = json.loads(match.group(1))
        start, end = match.span(2)
        if pred_text[start] == '"':
            start, end = start + 1, end - 1
        selected = {i for i, (left, right, _) in enumerate(token_spans)
                    if max(byte_offsets[start], left) < min(byte_offsets[end], right)}
        if selected:
            probabilities = [token_spans[i][2] for i in selected]
            predictions[index // 3][f"{field}_confidence"] = sum(probabilities) / len(probabilities)
            value_indices.update(selected)
    probabilities = [token_spans[i][2] for i in value_indices]
    return (predictions, sum(probabilities) / len(probabilities), min(probabilities)) if probabilities else (predictions, None, None)


def summarize_cases(cases):
    values = [case["value_min_confidence"] for case in cases if case["value_min_confidence"] is not None]
    averages = [case["value_avg_confidence"] for case in cases if case["value_avg_confidence"] is not None]
    correct = sum(case["is_exact_match"] for case in cases)
    return {
        "total": len(cases),
        "correct": correct,
        "accuracy": 100.0 * correct / len(cases) if cases else None,
        "value_min_confidence": min(values) if values else None,
        "value_avg_confidence": sum(averages) / len(averages) if averages else None,
        "confidence_count": len(values),
        "confidence_missing_count": len(cases) - len(values),
    }


## 6. 토크나이저와 4비트 모델


In [ ]:
def load_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(
        runtime_path("adapter_path"), **HYPER_PARAMS["tokenizer"],
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def load_model():
    dtype = resolve_dtype(HYPER_PARAMS["precision"]["compute_dtype"])
    embedding_dtype = resolve_dtype(HYPER_PARAMS["precision"]["frozen_embedding_dtype"])
    model = AutoModelForCausalLM.from_pretrained(
        RUNTIME_PARAMS["model_name"], revision=RUNTIME_PARAMS["model_revision"],
        dtype=dtype,
        quantization_config=BitsAndBytesConfig(
            bnb_4bit_compute_dtype=dtype, **HYPER_PARAMS["quantization"],
        ),
        **HYPER_PARAMS["model_loading"],
    )
    for module in model.modules():
        if "norm" in type(module).__name__.lower():
            module.to(dtype=torch.float32)
    for module in (model.get_input_embeddings(), model.get_output_embeddings()):
        module.to(dtype=embedding_dtype)
    model = PeftModel.from_pretrained(model, runtime_path("adapter_path"), is_trainable=False)
    model.config.use_cache = True
    model.eval()
    return model


In [ ]:
def prepare_evaluation(tokenizer):
    samples = read_jsonl(runtime_path("test_file"))
    prompts = [tokenizer.apply_chat_template(
        sample["messages"][:2], tokenize=False, add_generation_prompt=True,
        **HYPER_PARAMS["chat_template"],
    ) for sample in samples]
    paths = [runtime_path(key) for key in ("test_file", "split_meta_file", "run_config_file")]
    adapter = Path(runtime_path("adapter_path"))
    paths.extend(str(adapter / name) for name in
                 ("adapter_config.json", "adapter_model.safetensors", "tokenizer.json", "tokenizer_config.json"))
    return samples, prompts, {path: file_sha256(path) for path in paths}


## 7. 실행 정보와 평가


In [ ]:
def collect_execution_metadata(model=None):
    from importlib.metadata import PackageNotFoundError, version

    packages = {}
    for name in ("torch", "transformers", "peft", "accelerate", "bitsandbytes"):
        try:
            packages[name] = version(name)
        except PackageNotFoundError:
            packages[name] = None
    return {
        "python": platform.python_version(), "python_executable": sys.executable,
        "platform": platform.platform(), "packages": packages,
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "cuda": torch.version.cuda,
        "base_model": RUNTIME_PARAMS["model_name"],
        "base_model_revision": getattr(getattr(model, "config", None), "_commit_hash", None)
            or RUNTIME_PARAMS["model_revision"],
        "compute_dtype": str(resolve_dtype(HYPER_PARAMS["precision"]["compute_dtype"])),
        "input_embedding_dtype": str(model.get_input_embeddings().weight.dtype) if model else None,
        "output_embedding_dtype": str(model.get_output_embeddings().weight.dtype) if model else None,
        "gpu_total_bytes": torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None,
        "peak_allocated_bytes": torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None,
        "peak_reserved_bytes": torch.cuda.max_memory_reserved() if torch.cuda.is_available() else None,
    }


In [ ]:
def evaluate():
    report_path = Path(runtime_path("report_file"))
    if report_path.exists():
        raise FileExistsError(f"기존 평가 결과가 있습니다: {report_path}")
    settings = HYPER_PARAMS["evaluation"]
    tokenizer = load_tokenizer()
    samples, prompts, file_hashes = prepare_evaluation(tokenizer)
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    model = load_model()
    eos_ids = {tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")}
    eos_ids.discard(None)
    case_results = []
    print(f"평가 {len(samples)}건 / GPU: {torch.cuda.get_device_name(0)}")

    for start in tqdm(range(0, len(samples), settings["batch_size"]), desc="Evaluation"):
        batch = samples[start:start + settings["batch_size"]]
        inputs = tokenizer(prompts[start:start + len(batch)], return_tensors="pt", padding=True,
                           truncation=False, add_special_tokens=False).to(model.device)
        with torch.inference_mode(), torch.autocast(
                device_type="cuda", dtype=resolve_dtype(HYPER_PARAMS["precision"]["compute_dtype"])):
            outputs = model.generate(
                **inputs,
                max_new_tokens=settings["max_new_tokens"],
                do_sample=settings["do_sample"],
                temperature=None, top_p=None, top_k=None,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=sorted(eos_ids),
                return_dict_in_generate=True,
                output_scores=True,
            )
        generated = outputs.sequences[:, inputs["input_ids"].shape[1]:]
        for offset, sample in enumerate(batch):
            token_ids = generated[offset].tolist()
            end = next((i for i, token_id in enumerate(token_ids)
                        if token_id in eos_ids or token_id == tokenizer.pad_token_id), len(token_ids))
            terminated = end < len(token_ids) and token_ids[end] in eos_ids
            token_ids = token_ids[:end]
            pred_text = tokenizer.decode(token_ids, skip_special_tokens=False, clean_up_tokenization_spaces=False)
            probabilities = [torch.softmax(outputs.scores[i][offset].float(), dim=-1)[token_id].item()
                             for i, token_id in enumerate(token_ids)]
            token_spans = build_token_spans(tokenizer, token_ids, probabilities, pred_text)
            prediction, avg_confidence, min_confidence = extract_values_with_confidence(pred_text, token_spans)
            ground_truth = sample["messages"][2]["content"]
            norm_pred = parse_and_normalize(pred_text)
            is_match = norm_pred is not None and Counter(norm_pred) == Counter(parse_and_normalize(ground_truth))
            metadata = sample["metadata"]
            case_results.append({
                "case_id": start + offset + 1,
                **{key: metadata[key] for key in ("site", "image_id", "source_index", "test_type")},
                "input_user": sample["messages"][1]["content"],
                "ground_truth_raw": ground_truth,
                "prediction_raw": pred_text,
                "prediction": prediction,
                "is_valid_json_schema": norm_pred is not None,
                "is_exact_match": is_match,
                "output_limit_reached": not terminated and end >= settings["max_new_tokens"],
                "value_avg_confidence": avg_confidence,
                "value_min_confidence": min_confidence,
            })
        # 다음 배치를 생성하기 전에 어휘 점수를 해제합니다.
        del outputs, inputs, generated

    overall = summarize_cases(case_results)
    by_type = {name: summarize_cases([case for case in case_results if case["test_type"] == name])
               for name in sorted({case["test_type"] for case in case_results})}
    report = {
        "model": RUNTIME_PARAMS["model_name"],
        "adapter": runtime_path("adapter_path"),
        "device": torch.cuda.get_device_name(0),
        "dtype": str(model.dtype),
        "quantized": True, "adapter_merged": False,
        "duration_seconds": time.perf_counter() - started,
        "test_file": runtime_path("test_file"),
        "runtime_params": deepcopy(RUNTIME_PARAMS),
        "hyper_params": deepcopy(HYPER_PARAMS),
        "batch_size": settings["batch_size"],
        "scoring_version": "strict_schema_multiset_v1",
        "confidence_definition": "Generated tokens overlapping workType/spec/quantity value bytes; unique tokens per case. Group min includes correct and incorrect cases with known confidence; null cases are excluded and counted. Avg is the mean of known case averages.",
        "file_hashes": file_hashes,
        "execution": collect_execution_metadata(model),
        **overall,
        "correct_value_avg_confidence": summarize_cases([case for case in case_results if case["is_exact_match"]])["value_avg_confidence"],
        "incorrect_value_avg_confidence": summarize_cases([case for case in case_results if not case["is_exact_match"]])["value_avg_confidence"],
        "by_test_type": by_type,
        "cases": case_results,
    }
    write_json(report_path, report)
    print(f"완전 일치율: {overall['accuracy']:.2f}% ({overall['correct']}/{overall['total']})")
    print(f"평가 결과 저장: {report_path}")
    return report


## 8. 설정 확인 후 실행


In [ ]:
print(json.dumps(RUNTIME_PARAMS, ensure_ascii=False, indent=2))
print(json.dumps(HYPER_PARAMS, ensure_ascii=False, indent=2))


In [ ]:
report = evaluate()
